# Phase C v2: CCL Structural Pattern Recognition — CodeLlama 7B QLoRA

## Force is Oblivion / Hegemonikón Research (Lēthē)

**目的**: CCL (Cognitive Control Language) の構造的パターンを対照学習で獲得し、
Phase C-mini (CodeBERT 125M, ρ=0.963) をスケールアップ検証する。

| 項目 | 値 |
|:-----|:---|
| モデル | CodeLlama-7b-hf (4bit QLoRA) |
| データ | 1000ペア (500 positive + 500 negative, 4種) |
| 評価 | 5-fold CV, Spearman ρ, Accuracy, F1 |
| P14 検証 | L_Ξ ablation (λ=0 / 0.01 / 0.1 / 1.0) |
| GPU | Colab Pro+ A100 |

### 仮説

| # | 仮説 | 判定基準 |
|:--|:-----|:---------|
| P11' | CodeLlama QLoRA > Phase C-mini CodeBERT (ρ=0.963) | Δρ > 0 |
| P14 | L_Ξ あり > L_Ξ なし | Δρ > 0 at best λ |
| P41 | 7B モデルは hard negative を分離できる | hard_neg F1 > 0.7 |

In [ ]:
# Cell 1: 環境セットアップ
!nvidia-smi
!pip install -q transformers peft bitsandbytes accelerate datasets scipy scikit-learn

In [ ]:
# Cell 2: データアップロード
# phase_c_training_ccl.jsonl をアップロード (CCL 式のみ、軽量版)
from google.colab import files
import json

print("phase_c_training_ccl.jsonl をアップロードしてください")
uploaded = files.upload()

DATA_PATH = list(uploaded.keys())[0]
print(f"\nアップロード完了: {DATA_PATH}")

# データ読込 + 統計
records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))

n_pos = sum(1 for r in records if r['label'] == 1)
n_neg = sum(1 for r in records if r['label'] == 0)
types = {}
for r in records:
    t = r.get('pair_type', 'unknown')
    types[t] = types.get(t, 0) + 1

print(f"\n合計: {len(records)} ペア (positive: {n_pos}, negative: {n_neg})")
print(f"種別: {types}")

# cosine_49d の分布
import numpy as np
cos_pos = [r['cosine_49d'] for r in records if r['label'] == 1]
cos_neg = [r['cosine_49d'] for r in records if r['label'] == 0]
print(f"cosine_49d — positive: mean={np.mean(cos_pos):.3f}, std={np.std(cos_pos):.3f}")
print(f"cosine_49d — negative: mean={np.mean(cos_neg):.3f}, std={np.std(cos_neg):.3f}")

In [ ]:
# Cell 3: モデルロード (CodeLlama-7b, 4bit QLoRA)
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_NAME = "codellama/CodeLlama-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("トークナイザをロード中...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"モデルをロード中: {MODEL_NAME} (4bit)...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,  # 回帰 (similarity score)
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"パラメータ数: {model.num_parameters():,}")
print(f"GPU メモリ: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# QLoRA 設定
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU メモリ (LoRA 後): {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 4: データセット準備
from datasets import Dataset

MAX_LEN = 512

def format_pair(record):
    """CCL ペアをモデル入力に変換。"""
    text = f"Structure A: {record['anchor_ccl']}\n\nStructure B: {record['candidate_ccl']}"
    return {
        "text": text,
        "label": float(record['cosine_49d']),  # 回帰: 連続値
        "label_binary": record['label'],         # 分類: 0/1
        "pair_type": record.get('pair_type', 'unknown'),
    }

formatted = [format_pair(r) for r in records]
dataset = Dataset.from_list(formatted)

def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    tokens["label"] = batch["label"]
    return tokens

tokenized = dataset.map(tokenize, batched=True, batch_size=32, remove_columns=["text"])
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# トークン長の統計
lens = [len(tokenizer.encode(f["text"])) for f in formatted[:100]]
print(f"トークン長 (先頭100件): mean={np.mean(lens):.0f}, max={max(lens)}, >512: {sum(1 for l in lens if l > 512)}")
print(f"データセット: {len(tokenized)} 件")

In [ ]:
# Cell 5: L_Ξ カスタムトレーナー
#
# L_total = L_MSE + λ · L_Ξ
# L_Ξ = -Var(attention_weights) across heads
# = 不均一なアテンション分布を促進 (構造方向の選択的集中)

from transformers import Trainer, TrainingArguments
import torch.nn.functional as F_t

class LXiTrainer(Trainer):
    """L_Ξ 正則化付きトレーナー。

    λ > 0 のとき、アテンション重みの不均一性を促進する正則化項を追加。
    λ = 0 はベースライン (正則化なし)。
    """

    def __init__(self, *args, lxi_lambda: float = 0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.lxi_lambda = lxi_lambda

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels") if "labels" in inputs else inputs.pop("label")
        outputs = model(**inputs, output_attentions=(self.lxi_lambda > 0))
        logits = outputs.logits.squeeze(-1)

        # MSE 損失 (回帰)
        loss_mse = F_t.mse_loss(logits, labels.float())

        # L_Ξ: アテンション不均一性の促進
        if self.lxi_lambda > 0 and outputs.attentions is not None:
            # 最終層のアテンション重みを取得
            attn = outputs.attentions[-1]  # (B, H, S, S)
            # 各ヘッドのエントロピーの分散 = ヘッド間の特殊化度
            # エントロピーが低い = 集中的アテンション
            # ヘッド間の分散が大きい = 異なるヘッドが異なるパターンに集中
            attn_flat = attn.mean(dim=-1)  # (B, H, S) — 各位置のアテンション受容量
            head_var = attn_flat.var(dim=1).mean()  # ヘッド間の分散の平均
            loss_xi = -head_var  # 分散最大化 = 不均一促進
            loss = loss_mse + self.lxi_lambda * loss_xi
        else:
            loss = loss_mse

        return (loss, outputs) if return_outputs else loss

print("LXiTrainer 定義完了")

In [ ]:
# Cell 6: 5-fold CV + L_Ξ ablation
from sklearn.model_selection import StratifiedKFold
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, f1_score
from copy import deepcopy
import gc

N_FOLDS = 5
EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM = 4  # 実効バッチ = 16
LR = 2e-4

# L_Ξ 条件
LXI_CONDITIONS = [
    ("baseline", 0.0),
    ("lxi_0.01", 0.01),
    ("lxi_0.1", 0.1),
    ("lxi_1.0", 1.0),
]

# ラベル (fold 分割用の binary label)
binary_labels = np.array([r['label'] for r in records])
cosine_labels = np.array([r['cosine_49d'] for r in records])
pair_types = np.array([r.get('pair_type', 'unknown') for r in records])

all_results = {}

for cond_name, lxi_lambda in LXI_CONDITIONS:
    print(f"\n{'='*60}")
    print(f"  条件: {cond_name} (λ={lxi_lambda})")
    print(f"{'='*60}")

    fold_results = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(records)), binary_labels)):
        print(f"\n--- Fold {fold_idx+1}/{N_FOLDS} ---")

        train_ds = tokenized.select(train_idx.tolist())
        val_ds = tokenized.select(val_idx.tolist())

        # モデルを再初期化 (LoRA 重みのみリセット)
        # 注: 4bit モデルの deepcopy は重いので、LoRA のみリセット
        for name, param in model.named_parameters():
            if 'lora' in name and param.requires_grad:
                if 'weight' in name:
                    torch.nn.init.kaiming_uniform_(param)
                elif 'bias' in name:
                    torch.nn.init.zeros_(param)

        training_args = TrainingArguments(
            output_dir=f"/content/phase_c_{cond_name}_fold{fold_idx}",
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR,
            weight_decay=0.01,
            warmup_ratio=0.1,
            lr_scheduler_type="cosine",
            eval_strategy="epoch",
            save_strategy="no",  # メモリ節約
            bf16=True,
            logging_steps=20,
            report_to="none",
            seed=42 + fold_idx,
        )

        trainer = LXiTrainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            lxi_lambda=lxi_lambda,
        )

        trainer.train()

        # --- 評価 ---
        preds_out = trainer.predict(val_ds)
        pred_scores = preds_out.predictions.squeeze(-1)  # 予測 similarity
        true_cosine = cosine_labels[val_idx]
        true_binary = binary_labels[val_idx]
        val_types = pair_types[val_idx]

        # Spearman ρ (Phase C-mini と同じ指標)
        rho, p_val = spearmanr(pred_scores, true_cosine)

        # 二値分類 (閾値 = 予測スコアの中央値)
        threshold = np.median(pred_scores)
        pred_binary = (pred_scores >= threshold).astype(int)
        acc = accuracy_score(true_binary, pred_binary)
        f1 = f1_score(true_binary, pred_binary)

        # 種別別 F1
        type_f1 = {}
        for ptype in ['positive', 'easy_neg', 'hard_neg_cosine', 'hard_neg_dir']:
            mask = val_types == ptype
            if mask.sum() > 0:
                expected = np.ones(mask.sum()) if ptype == 'positive' else np.zeros(mask.sum())
                type_f1[ptype] = accuracy_score(expected.astype(int), pred_binary[mask])

        # MSE
        mse = float(np.mean((pred_scores - true_cosine) ** 2))

        fold_result = {
            "fold": fold_idx,
            "rho": float(rho),
            "p_val": float(p_val),
            "accuracy": float(acc),
            "f1": float(f1),
            "mse": float(mse),
            "type_accuracy": type_f1,
            "train_loss": preds_out.metrics.get("test_loss", None),
        }
        fold_results.append(fold_result)

        print(f"  ρ={rho:.4f}  acc={acc:.4f}  F1={f1:.4f}  MSE={mse:.4f}")
        for ptype, pacc in type_f1.items():
            print(f"    {ptype}: acc={pacc:.3f}")

        # メモリ解放
        del trainer
        gc.collect()
        torch.cuda.empty_cache()

    # 条件サマリー
    mean_rho = np.mean([r['rho'] for r in fold_results])
    mean_acc = np.mean([r['accuracy'] for r in fold_results])
    mean_f1 = np.mean([r['f1'] for r in fold_results])
    mean_mse = np.mean([r['mse'] for r in fold_results])

    all_results[cond_name] = {
        "lambda": lxi_lambda,
        "mean_rho": float(mean_rho),
        "mean_accuracy": float(mean_acc),
        "mean_f1": float(mean_f1),
        "mean_mse": float(mean_mse),
        "folds": fold_results,
    }

    print(f"\n  ★ {cond_name} 平均: ρ={mean_rho:.4f}  acc={mean_acc:.4f}  F1={mean_f1:.4f}  MSE={mean_mse:.4f}")

In [ ]:
# Cell 7: 結果サマリー + 仮説判定
print("=" * 70)
print("  Phase C v2 — 全条件サマリー")
print("=" * 70)
print(f"{'条件':<15} {'λ':>5} {'ρ':>8} {'acc':>8} {'F1':>8} {'MSE':>8}")
print("-" * 70)

# Phase C-mini 参照値
print(f"{'C-mini ref':<15} {'—':>5} {'0.963':>8} {'—':>8} {'—':>8} {'0.010':>8}")

for cond_name, data in all_results.items():
    print(f"{cond_name:<15} {data['lambda']:>5.2f} {data['mean_rho']:>8.4f} {data['mean_accuracy']:>8.4f} {data['mean_f1']:>8.4f} {data['mean_mse']:>8.4f}")

print("\n" + "=" * 70)
print("  仮説判定")
print("=" * 70)

baseline_rho = all_results['baseline']['mean_rho']
best_lxi = max(
    [(k, v) for k, v in all_results.items() if k != 'baseline'],
    key=lambda x: x[1]['mean_rho']
)

# P11': CodeLlama QLoRA vs Phase C-mini
delta_p11 = baseline_rho - 0.963
p11_verdict = "✅ 支持" if delta_p11 > 0 else "⚠️ 未達" if delta_p11 > -0.05 else "❌ 棄却"
print(f"P11': CodeLlama 7B QLoRA vs C-mini (ρ=0.963)")
print(f"  baseline ρ = {baseline_rho:.4f}, Δ = {delta_p11:+.4f} → {p11_verdict}")

# P14: L_Ξ ablation
delta_p14 = best_lxi[1]['mean_rho'] - baseline_rho
p14_verdict = "✅ 支持" if delta_p14 > 0.005 else "⚠️ 弱支持" if delta_p14 > 0 else "❌ 棄却"
print(f"\nP14: L_Ξ ablation")
print(f"  best = {best_lxi[0]} (ρ={best_lxi[1]['mean_rho']:.4f}), Δ vs baseline = {delta_p14:+.4f} → {p14_verdict}")

# P41: Hard negative 分離
hard_accs = []
for fold in all_results['baseline']['folds']:
    for t in ['hard_neg_cosine', 'hard_neg_dir']:
        if t in fold.get('type_accuracy', {}):
            hard_accs.append(fold['type_accuracy'][t])
if hard_accs:
    mean_hard = np.mean(hard_accs)
    p41_verdict = "✅ 支持" if mean_hard > 0.7 else "⚠️ 弱支持" if mean_hard > 0.5 else "❌ 棄却"
    print(f"\nP41: Hard negative 分離能力")
    print(f"  mean hard_neg accuracy = {mean_hard:.3f} → {p41_verdict}")

In [ ]:
# Cell 8: 結果保存 + ダウンロード
import json
from datetime import datetime

output = {
    "experiment": "phase_c_v2_qlora",
    "model": MODEL_NAME,
    "timestamp": datetime.now().isoformat(),
    "config": {
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "max_len": MAX_LEN,
        "lora_r": 16,
        "lora_alpha": 32,
        "n_pairs": len(records),
    },
    "reference": {
        "phase_c_mini_rho": 0.963,
        "phase_c_mini_model": "codebert-base (125M)",
        "phase_c_mini_pairs": 246,
    },
    "results": all_results,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}

output_path = "/content/phase_c_v2_results.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"結果保存: {output_path}")
files.download(output_path)

In [ ]:
# Cell 9: 可視化 (L_Ξ λ vs ρ)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

lambdas = [v['lambda'] for v in all_results.values()]
rhos = [v['mean_rho'] for v in all_results.values()]
accs = [v['mean_accuracy'] for v in all_results.values()]
mses = [v['mean_mse'] for v in all_results.values()]

# ρ vs λ
axes[0].plot(range(len(lambdas)), rhos, 'bo-', markersize=8)
axes[0].axhline(y=0.963, color='r', linestyle='--', label='C-mini (ρ=0.963)')
axes[0].set_xticks(range(len(lambdas)))
axes[0].set_xticklabels([f'λ={l}' for l in lambdas], rotation=45)
axes[0].set_ylabel('Spearman ρ')
axes[0].set_title('P14: L_Ξ λ vs ρ')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy vs λ
axes[1].plot(range(len(lambdas)), accs, 'go-', markersize=8)
axes[1].set_xticks(range(len(lambdas)))
axes[1].set_xticklabels([f'λ={l}' for l in lambdas], rotation=45)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs λ')
axes[1].grid(True, alpha=0.3)

# MSE vs λ
axes[2].plot(range(len(lambdas)), mses, 'rs-', markersize=8)
axes[2].set_xticks(range(len(lambdas)))
axes[2].set_xticklabels([f'λ={l}' for l in lambdas], rotation=45)
axes[2].set_ylabel('MSE')
axes[2].set_title('MSE vs λ')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/phase_c_v2_lxi_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
files.download('/content/phase_c_v2_lxi_ablation.png')
print("図保存完了")